In [7]:
import sys
from pathlib import Path

# Add the repo root (the folder containing src/) to sys.path
ROOT = Path.cwd()
while not (ROOT / "src" / "common.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.common import load_clean, slice_period

PERIOD = "2005-2015"  # must match a key in PERIODS in src/common.py

df = slice_period(load_clean(), PERIOD)

In [8]:
import pandas as pd
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "src" / "common.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.common import load_transformed, slice_period

PERIOD = "1990-2004"
df = slice_period(load_transformed(), PERIOD)
print(df.shape)

(180, 126)


In [9]:
# Missing-data audit for 1990-2004 (no dropping or filling here)
audit = pd.DataFrame({
    "n_missing": df.isna().sum(),
    "pct_missing": (100 * df.isna().mean()).round(1),
    "first_obs": df.apply(lambda s: s.first_valid_index()).dt.strftime("%Y-%m"),
    "last_obs": df.apply(lambda s: s.last_valid_index()).dt.strftime("%Y-%m"),
})

problem = audit[audit.n_missing > 0].sort_values("n_missing", ascending=False)
print(problem)
print("total missing cells:", int(df.isna().sum().sum()),
      "| complete series:", int((audit.n_missing == 0).sum()), "of", df.shape[1])
print("months with any NA:", df.index[df.isna().any(axis=1)].strftime("%Y-%m").tolist())

           n_missing  pct_missing first_obs last_obs
ACOGNO            26         14.4   1992-03  2004-12
M1SL               1          0.6   1990-01  2004-12
TOTRESNS           1          0.6   1990-01  2004-12
NONBORRES          1          0.6   1990-01  2004-12
total missing cells: 29 | complete series: 122 of 126
months with any NA: ['1990-01', '1990-02', '1990-03', '1990-04', '1990-05', '1990-06', '1990-07', '1990-08', '1990-09', '1990-10', '1990-11', '1990-12', '1991-01', '1991-02', '1991-03', '1991-04', '1991-05', '1991-06', '1991-07', '1991-08', '1991-09', '1991-10', '1991-11', '1991-12', '1992-01', '1992-02', '2001-10']


In [10]:
import numpy as np

# Rebuild transformed series from raw (no outlier removal) following FRED-MD tcodes
raw = pd.read_csv(ROOT / "data" / "raw" / "2026-09-MD.csv")
tcode = pd.to_numeric(raw.iloc[0, 1:]).astype(int)
x = raw.iloc[1:].copy()
x.index = pd.to_datetime(x["sasdate"], format="%m/%d/%Y")
x = x.drop(columns="sasdate").apply(pd.to_numeric, errors="coerce")

def transform(s, c):
    if c == 1: return s
    if c == 2: return s.diff()
    if c == 3: return s.diff().diff()
    if c == 4: return np.log(s)
    if c == 5: return np.log(s).diff()
    if c == 6: return np.log(s).diff().diff()
    if c == 7: return (s / s.shift(1) - 1).diff()

T = pd.DataFrame({v: transform(x[v], tcode[v]) for v in x.columns}).iloc[2:]
W = T.loc["1990-01-01":"2004-12-31"]

# 10 x IQR rule, statistics computed inside the 1990-2004 window only
med = W.median()
iqr = W.quantile(0.75) - W.quantile(0.25)
ratio = (W - med).abs() / iqr.where(iqr > 0)

hits = ratio.stack()
flagged = hits[hits > 10].reset_index()
flagged.columns = ["date", "series", "x_over_IQR"]
flagged["date"] = flagged["date"].dt.strftime("%Y-%m")
print(flagged.round(2).to_string(index=False))
print("n flagged:", len(flagged))
print("\nclosest below the threshold:")
print(hits.sort_values(ascending=False).head(6).round(2))

   date   series  x_over_IQR
2001-10     M1SL       12.96
2001-10 TOTRESNS       11.45
n flagged: 2

closest below the threshold:
sasdate                    
2001-10-01  M1SL               12.96
            TOTRESNS           11.45
            NONBORRES           9.74
            DSERRG3M086SBEA     9.66
2000-05-01  USGOVT              9.40
1992-12-01  RPI                 9.34
dtype: float64


In [11]:
results = ROOT / "periods" / "1990-2004" / "results"
results.mkdir(parents=True, exist_ok=True)

print("series with NA, no outlier removal:", W.columns[W.isna().any()].tolist())

# Descriptive statistics (transformed series, 1990-2004)
stats = pd.DataFrame({
    "tcode": tcode, "n": W.count(), "mean": W.mean(), "sd": W.std(),
    "skew": W.skew(), "kurt": W.kurt() + 3,
})
stats.round(4).to_csv(results / "descriptive_stats.csv")
print("\nfattest tails:")
print(stats.sort_values("kurt", ascending=False).head(6).round(3))

# Correlation structure (pairwise, so ACOGNO is kept)
C = W.corr()
upper = C.where(np.triu(np.ones(C.shape), k=1).astype(bool)).stack()
top = upper.reindex(upper.abs().sort_values(ascending=False).index)
top.head(50).round(3).to_csv(results / "top_correlated_pairs.csv")
print("\nmost correlated pairs:")
print(top.head(8).round(3))
print("share of pairs with |corr| > 0.5:", round(float((upper.abs() > 0.5).mean()), 3))

# How much does 2001-10 matter for the correlations?
W_no = W.drop(index=pd.Timestamp("2001-10-01"))
for a, b in [("M1SL", "TOTRESNS"), ("TOTRESNS", "NONBORRES")]:
    print(a, b, "with 2001-10:", round(W[a].corr(W[b]), 3),
          "| without:", round(W_no[a].corr(W_no[b]), 3))

series with NA, no outlier removal: ['ACOGNO']

fattest tails:
                 tcode    n   mean     sd   skew    kurt
TOTRESNS             6  180 -0.000  0.069 -3.415  44.186
M1SL                 6  180 -0.000  0.009 -3.351  42.843
NONBORRES            7  180 -0.000  0.064 -2.615  37.141
DSERRG3M086SBEA      6  180 -0.000  0.002  2.043  25.255
W875RX1              5  180  0.002  0.007  1.139  21.311
RPI                  5  180  0.003  0.006  1.514  21.084

most correlated pairs:
TOTRESNS    NONBORRES     0.994
AAAFFM      BAAFFM        0.991
PERMIT      PERMITS       0.990
RPI         W875RX1       0.990
WPSFD49207  WPSFD49502    0.988
IPFPNSS     IPFINAL       0.980
T5YFFM      T10YFFM       0.980
HOUST       HOUSTS        0.977
dtype: float64
share of pairs with |corr| > 0.5: 0.021
M1SL TOTRESNS with 2001-10: 0.675 | without: 0.396
TOTRESNS NONBORRES with 2001-10: 0.994 | without: 0.991


## Method

## Forecasting

## Evaluation

## Results